# Ropedia Academy — D2 · SLAM reuses Track B's geometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D2.ipynb)

> **Solves PnP by reprojection minimization and visualizes the convergence curve plus reprojected points landing on the observed detections.**
>
> 用重投影最小化求解 PnP，并可视化收敛曲线，以及重投影点落到观测检测上。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D2

In [ ]:
import torch, matplotlib.pyplot as plt

# SLAM = Track B geometry, online. TRACKING each frame = PnP (pose from 3D<->2D).
K = torch.tensor([[600,0,320.],[0,600,240.],[0,0,1.]])
pts3d = torch.randn(40, 3) + torch.tensor([0,0,4.])
R, t_true = torch.eye(3), torch.tensor([0.3,-0.2,0.1])
def project(P, t):
    p = P @ R.T + t; uv = (K @ p.T).T; return uv[:, :2] / uv[:, 2:3]
obs = project(pts3d, t_true)

t = torch.zeros(3, requires_grad=True); opt = torch.optim.Adam([t], lr=0.02); hist = []
for _ in range(300):                              # PnP by reprojection minimization
    opt.zero_grad(); loss = ((project(pts3d, t) - obs)**2).mean()
    loss.backward(); opt.step(); hist.append(loss.item())
print("recovered translation:", t.detach().round(decimals=2).tolist(), "| true:", t_true.tolist())

# Visualize PnP convergence and the predicted vs observed 2D projections.
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.3))
ax[0].plot(hist); ax[0].set_yscale('log'); ax[0].set_title("PnP convergence"); ax[0].set_xlabel("iter")
ax[1].scatter(*obs.T, c='g', s=12, label='observed'); ax[1].scatter(*project(pts3d, t).detach().T, c='r', marker='x', s=12, label='reprojected')
ax[1].legend(); ax[1].invert_yaxis(); ax[1].set_title("tracking = PnP (B geometry, online)"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D2
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks